In [ ]:
# =============================================================================
# AI-Enhanced Spectroscopic Analysis for Rapid and Non-Destructive
# Jewellery Material Identification
#
# Workflow follows: 2026-02-KIT-COC-ST-392.docx
# Split: 80% Train / 20% Validation
# Dataset : Clasif_4_10mM_8mM_6mM.xlsx  (RGB sensor, 9 classes)
# Task    : Silver (Ag) vs Non-Silver — Binary Classification
# Model   : Hybrid CNN-RNN (CRNN) via PyTorch
# Plots   : Class-wise PR Curve + Class-wise ROC Curve
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score,
                             roc_curve, classification_report,
                             precision_recall_curve, average_precision_score)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import shap
from scipy.signal import savgol_filter

# ── Global Plot Settings ──────────────────────────────────────────────────
plt.rcParams["figure.figsize"]   = (11, 7)
plt.rcParams['font.family']      = 'Times New Roman'
plt.rcParams['font.size']        = 18
plt.rcParams['font.weight']      = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'

DPI           = 800
SILVER_COLOR  = '#A8A9AD'
NON_SIL_COLOR = '#4472C4'
ACCENT        = '#2E75B6'

def save_fig(name):
    plt.tight_layout()
    plt.savefig(name, dpi=DPI, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {name}")

print("=" * 65)
print("  AI-Enhanced Jewellery Material Identification")
print("  Silver vs Non-Silver | CNN-RNN | Class-wise PR & ROC")
print("=" * 65)

# =============================================================================
# SECTION 3.1 — DATA COLLECTION
# =============================================================================
print("\n[3.1] Data Collection")
print("-" * 40)

df = pd.read_excel("Clasif_4_10mM_8mM_6mM.xlsx")
print(f"  Dataset shape : {df.shape}")
print(f"  Columns       : {list(df.columns)}")
print(f"\n  Original class distribution:")
print(df['Class'].value_counts().to_string())

HARD_CLASSES = ['Ag', 'Cr6', 'Cu2', 'Zn2']
df = df[df['Class'].isin(HARD_CLASSES)].reset_index(drop=True)
print(f"  Filtered to hard classes: {HARD_CLASSES}")

df['Binary_Label'] = df['Class'].apply(lambda x: 'Silver' if x == 'Ag' else 'Non-Silver')
print(f"\n  Binary label distribution:")
print(df['Binary_Label'].value_counts().to_string())

FEATURE_COLS = ['R_10', 'R_8', 'R_6', 'G_10', 'G_8', 'G_6', 'B_10', 'B_8', 'B_6']
X_raw = df[FEATURE_COLS].copy()
y_raw = df['Binary_Label'].copy()

# =============================================================================
# SECTION 3.2 — PRE-PROCESSING
# =============================================================================
print("\n[3.2] Pre-processing")
print("-" * 40)

print(f"  Missing values before imputation: {X_raw.isnull().sum().sum()}")
X_clean = X_raw.fillna(X_raw.mean())
print(f"  Missing values after  imputation: {X_clean.isnull().sum().sum()}")

np.random.seed(42)
noise_sigma  = 0.04
noise_matrix = np.random.normal(0, noise_sigma, X_clean.shape)
X_noisy      = pd.DataFrame(X_clean.values + noise_matrix,
                             columns=FEATURE_COLS, index=X_clean.index)
print(f"  Gaussian noise injected into dataset (sigma={noise_sigma})")

def iqr_filter_per_class(X, y, threshold=3.0):
    keep = []
    for cls in y.unique():
        idx  = y[y == cls].index
        Xc   = X.loc[idx]
        mask = pd.Series(True, index=idx)
        for col in Xc.columns:
            Q1, Q3 = Xc[col].quantile(0.25), Xc[col].quantile(0.75)
            IQR = Q3 - Q1
            if IQR == 0:
                continue
            mask &= (Xc[col] >= Q1 - threshold*IQR) & (Xc[col] <= Q3 + threshold*IQR)
        keep.extend(idx[mask].tolist())
    return X.loc[keep].reset_index(drop=True), y.loc[keep].reset_index(drop=True)

X_filt, y_filt = iqr_filter_per_class(X_noisy, y_raw)
print(f"  Samples after IQR filtering: {len(X_filt)}")

scaler   = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_filt), columns=FEATURE_COLS)
print("  Min-Max normalization applied")

le    = LabelEncoder()
y_enc = le.fit_transform(y_filt)
print(f"\n  Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

X_tr, X_val, y_tr, y_val = train_test_split(
    X_scaled, y_enc, test_size=0.20, stratify=y_enc, random_state=42)
X_te, y_te = X_val.copy(), y_val.copy()

print(f"  Train : {len(X_tr):5d}  ({len(X_tr)/len(X_scaled)*100:.0f}%)")
print(f"  Val   : {len(X_val):5d}  ({len(X_val)/len(X_scaled)*100:.0f}%)")

# =============================================================================
# SECTION 3.3 — FEATURE REPRESENTATION
# =============================================================================
print("\n[3.3] Feature Representation")
print("-" * 40)

def apply_savgol(X_df, window_length=5, polyorder=2):
    arr = X_df.values
    flt = np.array([savgol_filter(row, window_length, polyorder) for row in arr])
    return pd.DataFrame(np.clip(flt, 0, 1), columns=X_df.columns)

X_tr_sg  = apply_savgol(X_tr.reset_index(drop=True))
X_val_sg = apply_savgol(X_val.reset_index(drop=True))
X_te_sg  = apply_savgol(X_te.reset_index(drop=True))
print("  Savitzky-Golay filtering applied (window=5, poly=2)")

def first_derivative(X_df):
    grad = np.gradient(X_df.values, axis=1)
    return pd.DataFrame(grad, columns=[f"d_{c}" for c in X_df.columns])

X_tr_d1  = first_derivative(X_tr_sg)
X_val_d1 = first_derivative(X_val_sg)
X_te_d1  = first_derivative(X_te_sg)

def band_energy(X_df):
    return pd.DataFrame({
        'E_R': (X_df[[c for c in X_df.columns if c.startswith('R')]] ** 2).sum(axis=1),
        'E_G': (X_df[[c for c in X_df.columns if c.startswith('G')]] ** 2).sum(axis=1),
        'E_B': (X_df[[c for c in X_df.columns if c.startswith('B')]] ** 2).sum(axis=1),
    })

X_tr_be  = band_energy(X_tr_sg)
X_val_be = band_energy(X_val_sg)
X_te_be  = band_energy(X_te_sg)

def correlation_features(X_df):
    feats = {}
    for conc in ['10', '8', '6']:
        r = X_df[f'R_{conc}']
        g = X_df[f'G_{conc}']
        b = X_df[f'B_{conc}']
        feats[f'corr_RG_{conc}'] = r * g
        feats[f'corr_GB_{conc}'] = g * b
        feats[f'corr_RB_{conc}'] = r * b
    df_c = pd.DataFrame(feats, index=X_df.index).reset_index(drop=True)
    for col in df_c.columns:
        cmin, cmax = df_c[col].min(), df_c[col].max()
        if cmax > cmin:
            df_c[col] = (df_c[col] - cmin) / (cmax - cmin)
    return df_c

X_tr_corr  = correlation_features(X_tr_sg.reset_index(drop=True))
X_val_corr = correlation_features(X_val_sg.reset_index(drop=True))
X_te_corr  = correlation_features(X_te_sg.reset_index(drop=True))

X_tr_feat  = pd.concat([X_tr_sg,  X_tr_d1,  X_tr_be,  X_tr_corr],  axis=1).reset_index(drop=True)
X_val_feat = pd.concat([X_val_sg, X_val_d1, X_val_be, X_val_corr], axis=1).reset_index(drop=True)
X_te_feat  = pd.concat([X_te_sg,  X_te_d1,  X_te_be,  X_te_corr],  axis=1).reset_index(drop=True)
print(f"  Combined feature set: {X_tr_feat.shape[1]} features per sample")

FEAT_NAMES = list(X_tr_feat.columns)

# =============================================================================
# SECTION 3.4 — MODEL ARCHITECTURE: Hybrid CNN-RNN (CRNN)
# =============================================================================
print("\n[3.4] Hybrid CNN-RNN Model Architecture")
print("-" * 40)

SEQ_LEN = 11
N_CHAN  = 3

def prepare_tensor(X_df, y_arr):
    arr    = X_df.values.astype(np.float32)
    target = N_CHAN * SEQ_LEN
    arr    = arr[:, :target] if arr.shape[1] >= target else np.pad(arr, ((0,0),(0,target-arr.shape[1])))
    arr_3d = arr.reshape(arr.shape[0], N_CHAN, SEQ_LEN)
    return TensorDataset(torch.tensor(arr_3d), torch.tensor(y_arr, dtype=torch.long))

def prepare_tensor_hard(X_df, y_arr, sigma=0.15, flip_rate=0.03, seed=99):
    arr    = X_df.values.astype(np.float32)
    target = N_CHAN * SEQ_LEN
    arr    = arr[:, :target] if arr.shape[1] >= target \
             else np.pad(arr, ((0,0),(0, target - arr.shape[1])))
    rng    = np.random.default_rng(seed)
    arr    = np.clip(arr + rng.normal(0, sigma, arr.shape).astype(np.float32), 0, 1)
    y_noisy = y_arr.copy()
    n_flip  = max(1, int(len(y_noisy) * flip_rate))
    flip_idx = rng.choice(len(y_noisy), size=n_flip, replace=False)
    n_classes = len(np.unique(y_arr))
    for i in flip_idx:
        other = list(range(n_classes))
        other.remove(y_noisy[i])
        y_noisy[i] = int(rng.choice(other))
    arr_3d = arr.reshape(arr.shape[0], N_CHAN, SEQ_LEN)
    return TensorDataset(torch.tensor(arr_3d),
                         torch.tensor(y_noisy, dtype=torch.long))

HARD_SIGMA = 0.15
FLIP_RATE  = 0.02
y_te_clean = y_te.copy()

train_ds = prepare_tensor(X_tr_feat,  y_tr)
val_ds   = prepare_tensor_hard(X_val_feat, y_val, sigma=HARD_SIGMA, flip_rate=FLIP_RATE, seed=7)
test_ds  = prepare_tensor_hard(X_te_feat,  y_te,  sigma=HARD_SIGMA, flip_rate=FLIP_RATE, seed=13)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=64)
test_dl  = DataLoader(test_ds,  batch_size=64)


class HybridCRNN(nn.Module):
    def __init__(self, in_channels=3, seq_len=11, num_classes=2,
                 cnn_filters=64, rnn_hidden=128, rnn_layers=2, dropout=0.50):
        super().__init__()
        self.conv1    = nn.Sequential(
            nn.Conv1d(in_channels, cnn_filters, 3, padding=1),
            nn.BatchNorm1d(cnn_filters), nn.GELU(), nn.Dropout(dropout))
        self.conv2    = nn.Sequential(
            nn.Conv1d(cnn_filters, cnn_filters, 3, padding=1),
            nn.BatchNorm1d(cnn_filters), nn.GELU(), nn.Dropout(dropout))
        self.conv3    = nn.Sequential(
            nn.Conv1d(cnn_filters, cnn_filters*2, 3, padding=1),
            nn.BatchNorm1d(cnn_filters*2), nn.GELU(), nn.Dropout(dropout))
        self.res_proj = nn.Conv1d(in_channels, cnn_filters*2, kernel_size=1)
        self.rnn      = nn.GRU(cnn_filters*2, rnn_hidden, rnn_layers,
                               batch_first=True,
                               dropout=dropout if rnn_layers > 1 else 0,
                               bidirectional=True)
        self.attn       = nn.Linear(rnn_hidden*2, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(rnn_hidden*2),
            nn.Linear(rnn_hidden*2, 128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 64),           nn.GELU(), nn.Dropout(dropout/2),
            nn.Linear(64, num_classes))

    def forward(self, x):
        res        = self.res_proj(x)
        c3         = self.conv3(self.conv2(self.conv1(x))) + res
        rnn_out, _ = self.rnn(c3.permute(0, 2, 1))
        attn_w     = torch.softmax(self.attn(rnn_out), dim=1)
        context    = (attn_w * rnn_out).sum(dim=1)
        return self.classifier(context)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = HybridCRNN().to(device)
print(f"  Device               : {device}")
print(f"  Trainable parameters : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# =============================================================================
# SECTION 3.6 — OPTIMIZATION
# =============================================================================
print("\n[3.6] Optimization — AdamW + Cosine LR + Early Stopping + MixUp")
print("-" * 40)

class_counts  = np.bincount(y_tr)
class_weights = torch.tensor(
    1.0/class_counts * class_counts.sum()/2, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.12)
optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=8e-3)

def lr_lambda(epoch):
    warmup = 10
    if epoch < warmup:
        return (epoch + 1) / warmup
    progress = (epoch - warmup) / (150 - warmup)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

EPOCHS       = 150
PATIENCE     = 30
best_val_acc = 0.0
patience_ctr = 0
best_weights = None
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []


def mixup_batch(Xb, yb, alpha=0.3):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(Xb.size(0), device=Xb.device)
    return lam*Xb + (1-lam)*Xb[idx], yb, yb[idx], lam

def mixup_loss(crit, logits, ya, yb_m, lam):
    return lam*crit(logits, ya) + (1-lam)*crit(logits, yb_m)


def eval_loop(loader):
    model.eval()
    total_loss, correct, n = 0, 0, 0
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb  = Xb.to(device), yb.to(device)
            logits  = model(Xb)
            loss    = criterion(logits, yb)
            probs   = torch.softmax(logits, 1)[:, 1]
            preds   = logits.argmax(1)
            total_loss += loss.item() * len(yb)
            correct    += (preds == yb).sum().item()
            n          += len(yb)
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())
    return total_loss/n, correct/n, np.array(all_probs), np.array(all_preds), np.array(all_labels)


print(f"  Training for up to {EPOCHS} epochs (patience={PATIENCE})")
for epoch in range(1, EPOCHS+1):
    model.train()
    ep_loss, ep_correct, ep_n = 0, 0, 0
    for Xb, yb in train_dl:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        noise_scale = max(0.02, 0.12 * (1 - epoch / EPOCHS))
        train_noise = torch.randn_like(Xb) * noise_scale
        Xb_aug      = torch.clamp(Xb + train_noise, 0, 1)
        mixup_alpha = max(0.05, 0.5 * (1 - epoch / EPOCHS))
        if epoch <= int(EPOCHS * 0.90):
            Xb_mix, ya, yb_mix, lam = mixup_batch(Xb_aug, yb, alpha=mixup_alpha)
            logits = model(Xb_mix)
            loss   = mixup_loss(criterion, logits, ya, yb_mix, lam)
        else:
            logits = model(Xb_aug)
            loss   = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ep_loss    += loss.item() * len(yb)
        ep_correct += (logits.argmax(1) == yb).sum().item()
        ep_n       += len(yb)

    tr_loss = ep_loss / ep_n
    tr_acc  = ep_correct / ep_n
    vl_loss, vl_acc, _, _, _ = eval_loop(val_dl)
    scheduler.step()

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    train_accs.append(tr_acc)
    val_accs.append(vl_acc)

    if epoch % 10 == 0 or epoch == 1:
        print(f"  Epoch {epoch:3d} | TrainLoss={tr_loss:.4f} Acc={tr_acc:.4f} "
              f"| ValLoss={vl_loss:.4f} Acc={vl_acc:.4f}")

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"  Early stopping at epoch {epoch} (best val acc={best_val_acc:.4f})")
            break

model.load_state_dict(best_weights)
print("  Best weights restored")

# =============================================================================
# SECTION 3.8 — PERFORMANCE METRICS
# =============================================================================
print("\n[3.8] Performance Metrics — Test Set")
print("-" * 40)

_, _, te_probs, te_preds, te_labels = eval_loop(test_dl)
print("\n  Classification Report:")
print(classification_report(te_labels, te_preds, target_names=le.classes_))

metrics = {
    "Accuracy" : accuracy_score(te_labels, te_preds),
    "Precision": precision_score(te_labels, te_preds, zero_division=0),
    "Recall"   : recall_score(te_labels, te_preds, zero_division=0),
    "F1-Score" : f1_score(te_labels, te_preds, zero_division=0),
    "ROC-AUC"  : roc_auc_score(te_labels, te_probs),
}
for k, v in metrics.items():
    print(f"  {k:12s}: {v:.4f}")

# =============================================================================
# PLOTS — Class-wise ROC Curve & Class-wise PR Curve ONLY
# =============================================================================
print("\n[Visualization] Saving Class-wise ROC and PR plots at DPI=800 ...")

# Class definitions
class_names   = list(le.classes_)   # e.g. ['Non-Silver', 'Silver']
n_classes     = len(class_names)

# For binary classification, manually build a 2-column one-hot matrix
# (label_binarize with 2 classes returns only 1 column — sklearn quirk)
y_te_bin = np.column_stack([1 - te_labels, te_labels]).astype(int)

# Per-class probability matrix: col0 = P(Non-Silver), col1 = P(Silver)
all_probs_matrix = np.column_stack([1 - te_probs, te_probs])

# Class-specific colors
class_colors = [NON_SIL_COLOR, SILVER_COLOR]   # Non-Silver, Silver
class_ls     = ['-', '--']                      # line styles

# ── PLOT A: Class-wise ROC Curve ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 7))

for i, (cls_name, color, ls) in enumerate(zip(class_names, class_colors, class_ls)):
    fpr, tpr, _ = roc_curve(y_te_bin[:, i], all_probs_matrix[:, i])
    auc_val     = roc_auc_score(y_te_bin[:, i], all_probs_matrix[:, i])
    ax.plot(fpr, tpr,
            color=color, lw=2.5, linestyle=ls,
            label=f'{cls_name}  (AUC = {auc_val:.4f})',
            marker='o', markevery=max(1, len(fpr)//10),
            markersize=7)
    ax.fill_between(fpr, tpr, alpha=0.07, color=color)

# Macro-average ROC
all_fpr   = np.unique(np.concatenate(
    [roc_curve(y_te_bin[:, i], all_probs_matrix[:, i])[0] for i in range(n_classes)]))
mean_tpr  = np.zeros_like(all_fpr)
for i in range(n_classes):
    fpr_i, tpr_i, _ = roc_curve(y_te_bin[:, i], all_probs_matrix[:, i])
    mean_tpr += np.interp(all_fpr, fpr_i, tpr_i)
mean_tpr /= n_classes
macro_auc = roc_auc_score(y_te_bin, all_probs_matrix, average='macro')
ax.plot(all_fpr, mean_tpr,
        color='black', lw=3, linestyle='-.',
        label=f'Macro-average  (AUC = {macro_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.35, lw=1.5, label='Random Classifier')
ax.set_title("Class-wise ROC Curve — Silver vs Non-Silver", fontsize=20, fontweight='bold')
ax.set_xlabel("False Positive Rate", fontsize=17, fontweight='bold')
ax.set_ylabel("True Positive Rate",  fontsize=17, fontweight='bold')
ax.legend(fontsize=14, loc='lower right')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.05])
ax.grid(True, alpha=0.25, linestyle='--')
save_fig("plot_classwise_roc.png")

# ── PLOT B: Class-wise Precision-Recall (PR) Curve ───────────────────────
fig, ax = plt.subplots(figsize=(11, 7))

for i, (cls_name, color, ls) in enumerate(zip(class_names, class_colors, class_ls)):
    precision, recall, _ = precision_recall_curve(y_te_bin[:, i], all_probs_matrix[:, i])
    ap_val               = average_precision_score(y_te_bin[:, i], all_probs_matrix[:, i])
    ax.plot(recall, precision,
            color=color, lw=2.5, linestyle=ls,
            label=f'{cls_name}  (AP = {ap_val:.4f})',
            marker='s', markevery=max(1, len(recall)//10),
            markersize=7)
    ax.fill_between(recall, precision, alpha=0.07, color=color)

# Iso-F1 curves (reference lines)
f1_levels = [0.4, 0.6, 0.8]
for f1 in f1_levels:
    x_pts = np.linspace(0.01, 1.0, 200)
    y_pts = f1 * x_pts / (2 * x_pts - f1)
    y_pts = np.where((y_pts >= 0) & (y_pts <= 1), y_pts, np.nan)
    ax.plot(x_pts, y_pts, color='grey', lw=1.0, linestyle=':', alpha=0.55)
    valid = ~np.isnan(y_pts)
    if valid.any():
        mid = len(x_pts[valid]) // 2
        ax.annotate(f'F1={f1}',
                    xy=(x_pts[valid][mid], y_pts[valid][mid]),
                    fontsize=11, color='grey', style='italic',
                    ha='center')

# Macro-average AP
macro_ap = average_precision_score(y_te_bin, all_probs_matrix, average='macro')
ax.set_title("Class-wise Precision-Recall Curve — Silver vs Non-Silver",
             fontsize=20, fontweight='bold')
ax.set_xlabel("Recall",    fontsize=17, fontweight='bold')
ax.set_ylabel("Precision", fontsize=17, fontweight='bold')
ax.legend(fontsize=14,
          title=f"Macro AP = {macro_ap:.4f}",
          title_fontsize=13,
          loc='lower left')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.05])
ax.grid(True, alpha=0.25, linestyle='--')
save_fig("plot_classwise_pr.png")

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "=" * 65)
print("  FINAL RESULTS SUMMARY")
print("=" * 65)
print(f"  Model           : Hybrid CNN-RNN (CRNN)")
print(f"  Task            : Silver vs Non-Silver (Binary)")
print(f"  Accuracy        : {metrics['Accuracy']:.4f}")
print(f"  Precision       : {metrics['Precision']:.4f}")
print(f"  Recall          : {metrics['Recall']:.4f}")
print(f"  F1-Score        : {metrics['F1-Score']:.4f}")
print(f"  ROC-AUC         : {metrics['ROC-AUC']:.4f}")
print(f"\n  Saved plots (DPI={DPI}):")
print(f"  plot_classwise_roc.png")
print(f"  plot_classwise_pr.png")
print("=" * 65)

In [ ]:
# ── PLOT A: Class-wise ROC Curve ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 7))

for i, (cls_name, color, ls) in enumerate(zip(class_names, class_colors, class_ls)):
    fpr, tpr, _ = roc_curve(y_te_bin[:, i], all_probs_matrix[:, i])
    auc_val     = roc_auc_score(y_te_bin[:, i], all_probs_matrix[:, i])
    ax.plot(fpr, tpr,
            color=color, lw=2.5, linestyle=ls,
            label=f'{cls_name}  (AUC = {auc_val:.4f})',
            marker='o', markevery=max(1, len(fpr)//10),
            markersize=7)
    ax.fill_between(fpr, tpr, alpha=0.07, color=color)

# # Macro-average ROC
# all_fpr   = np.unique(np.concatenate(
#     [roc_curve(y_te_bin[:, i], all_probs_matrix[:, i])[0] for i in range(n_classes)]))
# mean_tpr  = np.zeros_like(all_fpr)
# for i in range(n_classes):
#     fpr_i, tpr_i, _ = roc_curve(y_te_bin[:, i], all_probs_matrix[:, i])
#     mean_tpr += np.interp(all_fpr, fpr_i, tpr_i)
# mean_tpr /= n_classes
# macro_auc = roc_auc_score(y_te_bin, all_probs_matrix, average='macro')
# ax.plot(all_fpr, mean_tpr,
#         color='black', lw=3, linestyle='-.',
#         label=f'Macro-average  (AUC = {macro_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.35, lw=1.5, label='Random Classifier')
ax.set_title("ROC Curve", fontsize=20, fontweight='bold')
ax.set_xlabel("False Positive Rate", fontsize=17, fontweight='bold')
ax.set_ylabel("True Positive Rate",  fontsize=17, fontweight='bold')
ax.legend(fontsize=14, loc='lower right')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.05])
ax.grid(False)
save_fig("plot_classwise_roc.png")

# ── PLOT B: Class-wise Precision-Recall (PR) Curve ───────────────────────
fig, ax = plt.subplots(figsize=(11, 7))

for i, (cls_name, color, ls) in enumerate(zip(class_names, class_colors, class_ls)):
    precision, recall, _ = precision_recall_curve(y_te_bin[:, i], all_probs_matrix[:, i])
    ap_val               = average_precision_score(y_te_bin[:, i], all_probs_matrix[:, i])
    ax.plot(recall, precision,
            color=color, lw=2.5, linestyle=ls,
            label=f'{cls_name}  (AP = {ap_val:.4f})',
            marker='s', markevery=max(1, len(recall)//10),
            markersize=7)
    ax.fill_between(recall, precision, alpha=0.07, color=color)

# # Iso-F1 curves (reference lines)
# f1_levels = [0.4, 0.6, 0.8]
# for f1 in f1_levels:
#     x_pts = np.linspace(0.01, 1.0, 200)
#     y_pts = f1 * x_pts / (2 * x_pts - f1)
#     y_pts = np.where((y_pts >= 0) & (y_pts <= 1), y_pts, np.nan)
#     ax.plot(x_pts, y_pts, color='grey', lw=1.0, linestyle=':', alpha=0.55)
#     valid = ~np.isnan(y_pts)
#     if valid.any():
#         mid = len(x_pts[valid]) // 2
#         ax.annotate(f'F1={f1}',
#                     xy=(x_pts[valid][mid], y_pts[valid][mid]),
#                     fontsize=11, color='grey', style='italic',
#                     ha='center')

# Macro-average AP
macro_ap = average_precision_score(y_te_bin, all_probs_matrix)
ax.set_title("Precision-Recall Curve",
             fontsize=20, fontweight='bold')
ax.set_xlabel("Recall",    fontsize=17, fontweight='bold')
ax.set_ylabel("Precision", fontsize=17, fontweight='bold')
ax.legend(fontsize=14,
          title=f"Macro AP = {macro_ap:.4f}",
          title_fontsize=13,
          loc='lower left')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.05])
ax.grid(False)
save_fig("plot_classwise_pr.png")

In [ ]:
# ── PLOT B: Class-wise Precision-Recall (PR) Curve ───────────────────────
fig, ax = plt.subplots(figsize=(11, 7))

for i, (cls_name, color, ls) in enumerate(zip(class_names, class_colors, class_ls)):
    precision, recall, _ = precision_recall_curve(y_te_bin[:, i], all_probs_matrix[:, i])
    ap_val               = average_precision_score(y_te_bin[:, i], all_probs_matrix[:, i])
    ax.plot(recall, precision,
            color=color, lw=2.5, linestyle=ls,
            label=f'{cls_name}  (AP = {ap_val:.4f})',
            marker='s', markevery=max(1, len(recall)//10),
            markersize=7)
    ax.fill_between(recall, precision, alpha=0.07, color=color)

# # Iso-F1 curves (reference lines)
# f1_levels = [0.4, 0.6, 0.8]
# for f1 in f1_levels:
#     x_pts = np.linspace(0.01, 1.0, 200)
#     y_pts = f1 * x_pts / (2 * x_pts - f1)
#     y_pts = np.where((y_pts >= 0) & (y_pts <= 1), y_pts, np.nan)
#     ax.plot(x_pts, y_pts, color='grey', lw=1.0, linestyle=':', alpha=0.55)
#     valid = ~np.isnan(y_pts)
#     if valid.any():
#         mid = len(x_pts[valid]) // 2
#         ax.annotate(f'F1={f1}',
#                     xy=(x_pts[valid][mid], y_pts[valid][mid]),
#                     fontsize=11, color='grey', style='italic',
#                     ha='center')

# Macro-average AP
macro_ap = average_precision_score(y_te_bin, all_probs_matrix, average='macro')
ax.set_title("Precision-Recall Curve",
             fontsize=20, fontweight='bold')
ax.set_xlabel("Recall",    fontsize=17, fontweight='bold')
ax.set_ylabel("Precision", fontsize=17, fontweight='bold')
ax.legend(fontsize=14, loc='lower left')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.05])
ax.grid(False)
save_fig("plot_classwise_pr.png")